32 — F5 Global Acceptance Artifact + Hard-Fail Freeze
Bu notebook yeni model eğitmez ve F5 sonucunu yeniden yorumlamaz.

Amaç, 1 Ekim denetiminde belirtilen kabul mantığı açığını kapatmaktır:

F5'in sekiz global kontrolünü kalıcı CSV/JSON artefact olarak kaydetmek,
2521 provenance_status=PASS bilgisini tek başına global F5 PASS yerine kullanmamak,
2521 benzersiz cycle_uid, satır hizası ve sayısal alanları ayrıca doğrulamak,
kullanılan F5 kodu ve kritik input dosyalarının SHA256 kimliğini kaydetmek,
future-perturbation testinin kapsamını açıkça 50 cycle olarak kaydetmek,
eksik/boş/yanlış protokol/başarısız kontrol durumunda validator'ın HARD FAIL verdiğini test etmek.
Beklenen protokol:

ONLINE_FIXED_START_CAUSAL_10S_V1

In [3]:

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import warnings
import shutil

import numpy as np
import pandas as pd
from scipy.io import loadmat

warnings.filterwarnings("ignore")

ROOT = Path(r"${PROJECT_ROOT}")
RESULTS = ROOT / "results"
DEPLOY = ROOT / "deployment" / "final_online_rf20x7"

MANIFEST_PATH = DEPLOY / "final_online_training_manifest.csv"
FEATURES_PATH = DEPLOY / "final_online_training_features.npz"
FEATURE_ORDER_PATH = DEPLOY / "feature_order.csv"

R16 = RESULTS / "16_online_fixed_start_causal_validation"
F4F5 = RESULTS / "28_F4_F5_reproducibility_github_release_prep"
REPO = F4F5 / "github_repo"

OUT = RESULTS / "29_F5_online_raw_provenance_reconstruction"
OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_ID = "ONLINE_FIXED_START_CAUSAL_10S_V1"
DURATION_S = 10.0
STABILIZATION_S = 5.0
EPS = 1e-9

for p in [MANIFEST_PATH, FEATURES_PATH, FEATURE_ORDER_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("OUTPUT:", OUT)


OUTPUT: ${PROJECT_ROOT}\results\29_F5_online_raw_provenance_reconstruction


In [4]:

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(1024 * 1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def first_col(df, candidates):
    m = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in m:
            return m[c.lower()]
    return None


def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0

    return float(np.corrcoef(a, b)[0, 1])


def signal_features(prefix, x, duration_s):
    x = np.asarray(x, dtype=float)

    t = np.linspace(
        0.0,
        float(duration_s),
        len(x),
    )

    if len(x) >= 2:
        slope = float(np.polyfit(t, x, 1)[0])
        diffs = np.diff(x)
        mean_abs_diff = float(np.mean(np.abs(diffs)))
        max_abs_diff = float(np.max(np.abs(diffs)))
    else:
        slope = 0.0
        mean_abs_diff = 0.0
        max_abs_diff = 0.0

    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x, ddof=0)),
        f"{prefix}_min": float(np.min(x)),
        f"{prefix}_max": float(np.max(x)),
        f"{prefix}_range": float(np.max(x) - np.min(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_start": float(x[0]),
        f"{prefix}_end": float(x[-1]),
        f"{prefix}_delta": float(x[-1] - x[0]),
        f"{prefix}_slope_per_s": slope,
        f"{prefix}_mean_abs_diff": mean_abs_diff,
        f"{prefix}_max_abs_diff": max_abs_diff,
    }


def extract_44(v, i, temp, duration_s=10.0):
    feats = {}

    feats.update(signal_features("v", v, duration_s))
    feats.update(signal_features("i", i, duration_s))
    feats.update(signal_features("t", temp, duration_s))

    i = np.asarray(i, dtype=float)
    v = np.asarray(v, dtype=float)
    temp = np.asarray(temp, dtype=float)

    discharge_current = np.maximum(-i, 0.0)
    discharge_power = v * discharge_current

    feats.update({
        "i_rms": float(np.sqrt(np.mean(i ** 2))),
        "i_abs_mean": float(np.mean(np.abs(i))),
        "i_active_fraction": float(np.mean(np.abs(i) > 0.1)),
        "discharge_power_mean_w": float(np.mean(discharge_power)),
        "discharge_power_std_w": float(np.std(discharge_power, ddof=0)),
        "discharge_power_max_w": float(np.max(discharge_power)),
        "v_i_corr": safe_corr(v, i),
        "temperature_rise_c": float(temp[-1] - temp[0]),
    })

    return feats


In [5]:

manifest = pd.read_csv(MANIFEST_PATH)
manifest["cycle_uid"] = manifest["cycle_uid"].astype(str)
manifest["battery_id"] = manifest["battery_id"].astype(str)
manifest["_manifest_row"] = np.arange(len(manifest), dtype=int)

npz = np.load(FEATURES_PATH, allow_pickle=False)
X = np.asarray(npz["X_features"], dtype=np.float32)

feature_order = (
    pd.read_csv(FEATURE_ORDER_PATH, encoding="utf-8-sig")
    .sort_values("feature_index")["feature_name"]
    .astype(str)
    .tolist()
)

if X.shape != (2521, 44):
    raise RuntimeError(X.shape)

if len(feature_order) != 44:
    raise RuntimeError(f"Expected 44 feature names; got {len(feature_order)}")

test_dict = extract_44(
    np.ones(11),
    -np.ones(11),
    np.ones(11) * 25.0,
    10.0,
)

missing_feature_names = sorted(
    set(feature_order) - set(test_dict.keys())
)

if missing_feature_names:
    raise RuntimeError(
        f"Feature extractor names do not cover canonical order: {missing_feature_names}"
    )

print("Manifest rows:", len(manifest))
print("Batteries:", manifest["battery_id"].nunique())
print("Feature order:", feature_order)


Manifest rows: 2521
Batteries: 34
Feature order: ['v_mean', 'v_std', 'v_min', 'v_max', 'v_range', 'v_median', 'v_start', 'v_end', 'v_delta', 'v_slope_per_s', 'v_mean_abs_diff', 'v_max_abs_diff', 'i_mean', 'i_std', 'i_min', 'i_max', 'i_range', 'i_median', 'i_start', 'i_end', 'i_delta', 'i_slope_per_s', 'i_mean_abs_diff', 'i_max_abs_diff', 't_mean', 't_std', 't_min', 't_max', 't_range', 't_median', 't_start', 't_end', 't_delta', 't_slope_per_s', 't_mean_abs_diff', 't_max_abs_diff', 'i_rms', 'i_abs_mean', 'i_active_fraction', 'discharge_power_mean_w', 'discharge_power_std_w', 'discharge_power_max_w', 'v_i_corr', 'temperature_rise_c']


In [6]:

frames = [("final_manifest", manifest.copy())]

if R16.exists():
    for p in sorted(R16.rglob("*.csv")):
        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        if "cycle_uid" in df.columns:
            df["cycle_uid"] = df["cycle_uid"].astype(str)
            frames.append((p.name, df))

timing_aliases = {
    "decision_time_s": [
        "decision_time_s",
        "decision_time",
        "decision_timestamp_s",
        "online_decision_time_s",
    ],
    "active_start_time_s": [
        "active_start_time_s",
        "active_discharge_start_s",
        "active_start_s",
    ],
    "decision_delay_s": [
        "decision_delay_s",
        "decision_wait_s",
        "wait_after_target_s",
    ],
    "window_start_s": [
        "window_start_s",
        "start_time_s",
        "window_t0_s",
    ],
    "window_end_s": [
        "window_end_s",
        "end_time_s",
        "window_t1_s",
    ],
    "future_support_s": [
        "future_support_s",
        "future_support_max_s",
    ],
}

timing = manifest[["cycle_uid", "battery_id", "_manifest_row"]].copy()
source_rows = []

for semantic, aliases in timing_aliases.items():
    found = None

    for source_name, df in frames:
        c = first_col(df, aliases)
        if c is None:
            continue

        tmp = df[["cycle_uid", c]].drop_duplicates("cycle_uid")

        if tmp["cycle_uid"].duplicated().any():
            continue

        found = (source_name, c, tmp)
        break

    if found is None:
        timing[semantic] = np.nan
        source_rows.append({
            "field": semantic,
            "source": None,
            "column": None,
            "found": False,
        })
    else:
        source_name, c, tmp = found
        tmp = tmp.rename(columns={c: semantic})

        timing = timing.merge(
            tmp,
            on="cycle_uid",
            how="left",
            validate="one_to_one",
        )

        source_rows.append({
            "field": semantic,
            "source": source_name,
            "column": c,
            "found": True,
        })

timing_sources = pd.DataFrame(source_rows)
display(timing_sources)

if timing["decision_time_s"].isna().any():
    raise RuntimeError(
        "decision_time_s is required for F5 provenance reconstruction."
    )

# Resolve nominal window start/end if not directly stored.
if timing["window_start_s"].isna().all():
    if timing["active_start_time_s"].notna().all():
        timing["window_start_s"] = (
            pd.to_numeric(timing["active_start_time_s"])
            + STABILIZATION_S
        )
        window_start_formula = "active_start_time_s + 5 s"
    elif timing["decision_delay_s"].notna().all():
        timing["window_end_s"] = (
            pd.to_numeric(timing["decision_time_s"])
            - pd.to_numeric(timing["decision_delay_s"])
        )
        timing["window_start_s"] = timing["window_end_s"] - DURATION_S
        window_start_formula = "decision_time_s - decision_delay_s - 10 s"
    else:
        raise RuntimeError(
            "Cannot reconstruct nominal window_start_s from available metadata."
        )
else:
    window_start_formula = "stored window_start_s"

if timing["window_end_s"].isna().all():
    timing["window_end_s"] = (
        pd.to_numeric(timing["window_start_s"])
        + DURATION_S
    )

print("Window start source:", window_start_formula)
print(
    "Decision time coverage:",
    timing["decision_time_s"].notna().sum(),
    "/",
    len(timing),
)


,field,source,column,found
0,decision_time_s,final_manifest,decision_time_s,True
1,active_start_time_s,None,None,False
2,decision_delay_s,None,None,False
3,window_start_s,final_manifest,window_start_s,True
4,window_end_s,final_manifest,window_end_s,True
5,future_support_s,final_manifest,future_support_s,True


Window start source: stored window_start_s
Decision time coverage: 2521 / 2521


In [7]:


search_roots = [
    ROOT.parent,
    ROOT,
]

# Add explicit raw/data roots if present.
for p in [
    ROOT.parent / "data",
    ROOT / "data",
    ROOT.parent / "dataset",
    ROOT / "dataset",
]:
    if p.exists():
        search_roots.append(p)

all_mat = []

for base in search_roots:
    if not base.exists():
        continue

    for p in base.rglob("*.mat"):
        all_mat.append(p)

# Deduplicate
seen = set()
all_mat = [
    p for p in all_mat
    if not (str(p.resolve()) in seen or seen.add(str(p.resolve())))
]

battery_to_raw = {}

for battery in sorted(manifest["battery_id"].unique()):
    exact = [
        p for p in all_mat
        if p.stem.upper() == battery.upper()
    ]

    if len(exact) == 1:
        battery_to_raw[battery] = exact[0]
    elif len(exact) > 1:
        # Prefer shortest path / simplest source.
        exact = sorted(
            exact,
            key=lambda x: (len(str(x)), str(x)),
        )
        battery_to_raw[battery] = exact[0]
    else:
        battery_to_raw[battery] = None

raw_map = pd.DataFrame([
    {
        "battery_id": b,
        "raw_path": str(p) if p is not None else None,
        "found": p is not None,
    }
    for b, p in battery_to_raw.items()
])

display(raw_map)

if not raw_map["found"].all():
    print(
        "Missing raw battery files:",
        raw_map.loc[~raw_map["found"], "battery_id"].tolist(),
    )


,battery_id,raw_path,found
0,B0005,${USER_HOME}\Desktop\battery soh short windo...,True
1,B0006,${USER_HOME}\Desktop\battery soh short windo...,True
2,B0007,${USER_HOME}\Desktop\battery soh short windo...,True
3,B0018,${USER_HOME}\Desktop\battery soh short windo...,True
4,B0025,${USER_HOME}\Desktop\battery soh short windo...,True
5,B0026,${USER_HOME}\Desktop\battery soh short windo...,True
6,B0027,${USER_HOME}\Desktop\battery soh short windo...,True
7,B0028,${USER_HOME}\Desktop\battery soh short windo...,True
8,B0029,${USER_HOME}\Desktop\battery soh short windo...,True
9,B0030,${USER_HOME}\Desktop\battery soh short windo...,True


In [8]:

def get_field(obj, name):
    if hasattr(obj, name):
        return getattr(obj, name)

    if isinstance(obj, np.void) and obj.dtype.names and name in obj.dtype.names:
        return obj[name]

    if isinstance(obj, dict) and name in obj:
        return obj[name]

    raise AttributeError(name)


def scalar_text(x):
    if isinstance(x, bytes):
        return x.decode(errors="ignore")
    if isinstance(x, str):
        return x

    arr = np.asarray(x)

    if arr.size == 1:
        return str(arr.reshape(-1)[0])

    return str(x)


def to_1d_float(x):
    arr = np.asarray(x, dtype=float)
    return arr.reshape(-1)


def load_discharge_cycles(path):
    mat = loadmat(
        path,
        squeeze_me=True,
        struct_as_record=False,
    )

    if "cycle" in mat:
        cycles = np.atleast_1d(mat["cycle"]).reshape(-1)
    else:
        # Fallback: search any top-level object with cycle field.
        cycles = None

        for key, value in mat.items():
            if key.startswith("__"):
                continue

            try:
                candidate = get_field(value, "cycle")
                cycles = np.atleast_1d(candidate).reshape(-1)
                break
            except Exception:
                pass

        if cycles is None:
            raise RuntimeError(
                f"Could not find top-level cycle in {path}"
            )

    out = []

    for raw_idx, cyc in enumerate(cycles):
        try:
            typ = scalar_text(
                get_field(cyc, "type")
            ).strip().lower()
        except Exception:
            continue

        if "discharge" not in typ:
            continue

        try:
            data = get_field(cyc, "data")

            t = to_1d_float(
                get_field(data, "Time")
            )

            v = to_1d_float(
                get_field(data, "Voltage_measured")
            )

            i = to_1d_float(
                get_field(data, "Current_measured")
            )

            temp = to_1d_float(
                get_field(data, "Temperature_measured")
            )

            try:
                cap = float(
                    np.asarray(
                        get_field(data, "Capacity")
                    ).reshape(-1)[0]
                )
            except Exception:
                cap = np.nan

        except Exception:
            continue

        n = min(
            len(t),
            len(v),
            len(i),
            len(temp),
        )

        t = t[:n]
        v = v[:n]
        i = i[:n]
        temp = temp[:n]

        order = np.argsort(t)

        t = t[order]
        v = v[order]
        i = i[order]
        temp = temp[order]

        out.append({
            "raw_cycle_array_index": int(raw_idx),
            "discharge_sequence_index": len(out),
            "time": t,
            "voltage": v,
            "current": i,
            "temperature": temp,
            "capacity_ah": cap,
        })

    return out


cycle_cache = {}

def get_battery_cycles(battery_id):
    if battery_id in cycle_cache:
        return cycle_cache[battery_id]

    path = battery_to_raw[battery_id]

    if path is None:
        raise FileNotFoundError(
            f"No raw .mat file for {battery_id}"
        )

    cycles = load_discharge_cycles(path)
    cycle_cache[battery_id] = cycles
    return cycles


# Quick structural smoke test on first available battery.
first_battery = next(
    b for b, p in battery_to_raw.items()
    if p is not None
)

smoke = get_battery_cycles(first_battery)

print(
    "Smoke battery:",
    first_battery,
    "discharge cycles:",
    len(smoke),
)


Smoke battery: B0005 discharge cycles: 168


In [9]:

# We match by the raw Capacity target implied by SOH:
# Capacity = SOH/100 * 2.0 Ah.
# Then chronological order is preserved within each battery.

mapping_rows = []

for battery in sorted(manifest["battery_id"].unique()):

    raw_path = battery_to_raw.get(battery)

    bman = (
        manifest[
            manifest["battery_id"] == battery
        ]
        .sort_values("_manifest_row")
        .copy()
    )

    if raw_path is None:
        for _, row in bman.iterrows():
            mapping_rows.append({
                "cycle_uid": row["cycle_uid"],
                "battery_id": battery,
                "mapping_status": "raw_file_missing",
                "raw_cycle_array_index": np.nan,
                "discharge_sequence_index": np.nan,
                "raw_capacity_ah": np.nan,
                "target_capacity_ah": float(row["soh_rated_pct"]) / 100.0 * 2.0,
                "capacity_abs_diff_ah": np.nan,
            })
        continue

    cycles = get_battery_cycles(battery)

    capacities = np.asarray(
        [c["capacity_ah"] for c in cycles],
        dtype=float,
    )

    previous_idx = -1

    for _, row in bman.iterrows():

        target_cap = (
            float(row["soh_rated_pct"])
            / 100.0
            * 2.0
        )

        candidates = np.arange(
            previous_idx + 1,
            len(cycles),
        )

        if len(candidates) == 0:
            mapping_rows.append({
                "cycle_uid": row["cycle_uid"],
                "battery_id": battery,
                "mapping_status": "no_remaining_discharge_cycle",
                "raw_cycle_array_index": np.nan,
                "discharge_sequence_index": np.nan,
                "raw_capacity_ah": np.nan,
                "target_capacity_ah": target_cap,
                "capacity_abs_diff_ah": np.nan,
            })
            continue

        diffs = np.abs(
            capacities[candidates]
            - target_cap
        )

        best_pos = int(
            candidates[
                np.nanargmin(diffs)
            ]
        )

        best_diff = float(
            abs(
                capacities[best_pos]
                - target_cap
            )
        )

        # Capacity originates from raw NASA Capacity, so normally this is tiny.
        status = (
            "matched"
            if best_diff <= 1e-8
            else "matched_capacity_warning"
        )

        cyc = cycles[best_pos]

        mapping_rows.append({
            "cycle_uid": row["cycle_uid"],
            "battery_id": battery,
            "mapping_status": status,
            "raw_cycle_array_index": cyc["raw_cycle_array_index"],
            "discharge_sequence_index": cyc["discharge_sequence_index"],
            "raw_capacity_ah": cyc["capacity_ah"],
            "target_capacity_ah": target_cap,
            "capacity_abs_diff_ah": best_diff,
        })

        previous_idx = best_pos


cycle_map = pd.DataFrame(mapping_rows)

display(
    cycle_map.groupby(
        "mapping_status"
    ).size().rename("rows")
)

print(
    "Max capacity match abs diff Ah:",
    cycle_map["capacity_abs_diff_ah"].max(),
)


mapping_status
matched    2521
Name: rows, dtype: int64

Max capacity match abs diff Ah: 6.661338147750939e-16


In [10]:

def interpolation_support_indices(
    raw_t,
    decision_time,
    grid,
):
    raw_t = np.asarray(raw_t, dtype=float)

    available_idx = np.where(
        raw_t <= decision_time + EPS
    )[0]

    if len(available_idx) == 0:
        return np.asarray([], dtype=int)

    t_available = raw_t[available_idx]

    used_local = set()

    for q in grid:

        j = int(
            np.searchsorted(
                t_available,
                q,
                side="left",
            )
        )

        if j <= 0:
            used_local.add(0)

        elif j >= len(t_available):
            used_local.add(
                len(t_available) - 1
            )

        elif abs(
            t_available[j] - q
        ) <= 1e-12:
            used_local.add(j)

        else:
            used_local.add(j - 1)
            used_local.add(j)

    used_local = np.asarray(
        sorted(used_local),
        dtype=int,
    )

    return available_idx[used_local]


def reconstruct_window(
    cycle,
    window_start,
    window_end,
    decision_time,
):
    raw_t = cycle["time"]
    v = cycle["voltage"]
    i = cycle["current"]
    temp = cycle["temperature"]

    available = raw_t <= decision_time + EPS

    if available.sum() == 0:
        raise RuntimeError("No causal raw samples available.")

    t_avail = raw_t[available]
    v_avail = v[available]
    i_avail = i[available]
    temp_avail = temp[available]

    grid = np.linspace(
        window_start,
        window_end,
        11,
    )

    # IMPORTANT — canonical online pipeline parity:
    # np.interp returns float64. The frozen online training pipeline quantized
    # the 11-point V/I/T grid to float32 before 44-feature extraction.
    # Notebook 29B showed that this exact cast removes the residual v_i_corr
    # mismatch and gives 2521/2521 parity at the 1e-5 feature tolerance.
    v_grid = np.interp(
        grid,
        t_avail,
        v_avail,
    ).astype(np.float32)

    i_grid = np.interp(
        grid,
        t_avail,
        i_avail,
    ).astype(np.float32)

    temp_grid = np.interp(
        grid,
        t_avail,
        temp_avail,
    ).astype(np.float32)

    support_idx = interpolation_support_indices(
        raw_t,
        decision_time,
        grid,
    )

    return (
        grid,
        v_grid,
        i_grid,
        temp_grid,
        support_idx,
    )


work = (
    timing[
        [
            "cycle_uid",
            "battery_id",
            "_manifest_row",
            "decision_time_s",
            "window_start_s",
            "window_end_s",
            "future_support_s",
        ]
    ]
    .merge(
        cycle_map,
        on=[
            "cycle_uid",
            "battery_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

provenance_rows = []
feature_diff_rows = []

raw_file_hash_cache = {}

for n, row in work.iterrows():

    battery = row["battery_id"]
    raw_path = battery_to_raw.get(battery)

    base = {
        "cycle_uid": row["cycle_uid"],
        "battery_id": battery,
        "manifest_row": int(row["_manifest_row"]),
        "protocol_id": PROTOCOL_ID,
    }

    if (
        raw_path is None
        or pd.isna(
            row["discharge_sequence_index"]
        )
    ):
        base.update({
            "provenance_status": "FAIL_MAPPING",
            "raw_source_file": None,
            "raw_source_sha256": None,
            "raw_cycle_array_index": row["raw_cycle_array_index"],
            "discharge_sequence_index": row["discharge_sequence_index"],
            "decision_time_s": row["decision_time_s"],
            "window_start_s": row["window_start_s"],
            "window_end_s": row["window_end_s"],
            "raw_sample_indices": None,
            "latest_used_timestamp_s": np.nan,
            "feature_max_abs_diff": np.nan,
        })

        provenance_rows.append(base)
        continue

    if str(raw_path.resolve()) not in raw_file_hash_cache:
        raw_file_hash_cache[
            str(raw_path.resolve())
        ] = sha256_file(raw_path)

    cycles = get_battery_cycles(battery)

    cidx = int(
        row["discharge_sequence_index"]
    )

    cycle = cycles[cidx]

    try:
        (
            grid,
            vg,
            ig,
            tg,
            support_idx,
        ) = reconstruct_window(
            cycle,
            float(row["window_start_s"]),
            float(row["window_end_s"]),
            float(row["decision_time_s"]),
        )

        fdict = extract_44(
            vg,
            ig,
            tg,
            DURATION_S,
        )

        fvec = np.asarray(
            [
                fdict[name]
                for name in feature_order
            ],
            dtype=np.float32,
        )

        expected = X[
            int(row["_manifest_row"])
        ]

        max_diff = float(
            np.max(
                np.abs(
                    fvec.astype(np.float64)
                    - expected.astype(np.float64)
                )
            )
        )

        latest_used = (
            float(
                np.max(
                    cycle["time"][
                        support_idx
                    ]
                )
            )
            if len(support_idx)
            else np.nan
        )

        raw_inside = int(
            np.sum(
                (
                    cycle["time"]
                    >= float(
                        row["window_start_s"]
                    )
                    - EPS
                )
                &
                (
                    cycle["time"]
                    <= float(
                        row["window_end_s"]
                    )
                    + EPS
                )
            )
        )

        status = (
            "PASS"
            if max_diff <= 1e-5
            else "FAIL_FEATURE_PARITY"
        )

        base.update({
            "provenance_status": status,
            "raw_source_file": raw_path.name,
            "raw_source_sha256": raw_file_hash_cache[
                str(raw_path.resolve())
            ],
            "raw_cycle_array_index": int(
                cycle[
                    "raw_cycle_array_index"
                ]
            ),
            "discharge_sequence_index": int(
                cycle[
                    "discharge_sequence_index"
                ]
            ),
            "decision_time_s": float(
                row["decision_time_s"]
            ),
            "window_start_s": float(
                row["window_start_s"]
            ),
            "window_end_s": float(
                row["window_end_s"]
            ),
            "raw_samples_inside_nominal_interval": raw_inside,
            "raw_sample_indices": ";".join(
                map(str, support_idx.tolist())
            ),
            "raw_sample_count_used_for_interp": int(
                len(support_idx)
            ),
            "latest_used_timestamp_s": latest_used,
            "future_support_s": row["future_support_s"],
            "feature_max_abs_diff": max_diff,
            "feature_sha256_float32_le": hashlib.sha256(
                np.asarray(
                    expected,
                    dtype="<f4",
                ).tobytes(order="C")
            ).hexdigest(),
            "target_source": "final_online_training_manifest.csv::soh_rated_pct",
        })

        provenance_rows.append(base)

        feature_diff_rows.append({
            "cycle_uid": row["cycle_uid"],
            "battery_id": battery,
            "max_abs_diff": max_diff,
        })

    except Exception as e:
        base.update({
            "provenance_status": f"FAIL_EXCEPTION:{type(e).__name__}",
            "raw_source_file": raw_path.name,
            "raw_source_sha256": raw_file_hash_cache[
                str(raw_path.resolve())
            ],
            "raw_cycle_array_index": row["raw_cycle_array_index"],
            "discharge_sequence_index": row["discharge_sequence_index"],
            "decision_time_s": row["decision_time_s"],
            "window_start_s": row["window_start_s"],
            "window_end_s": row["window_end_s"],
            "raw_sample_indices": None,
            "latest_used_timestamp_s": np.nan,
            "feature_max_abs_diff": np.nan,
        })
        provenance_rows.append(base)


prov = pd.DataFrame(provenance_rows)
feature_diffs = pd.DataFrame(feature_diff_rows)

display(
    prov.groupby(
        "provenance_status"
    ).size().rename("rows")
)

if len(feature_diffs):
    print(
        "Global raw→44 feature max abs diff:",
        feature_diffs["max_abs_diff"].max(),
    )


provenance_status
PASS    2521
Name: rows, dtype: int64

Global raw→44 feature max abs diff: 0.0


In [11]:

# Gate 1: latest used raw timestamp must not exceed decision time.
good = prov[
    prov["provenance_status"] == "PASS"
].copy()

if len(good):
    good["latest_minus_decision_s"] = (
        pd.to_numeric(
            good["latest_used_timestamp_s"]
        )
        - pd.to_numeric(
            good["decision_time_s"]
        )
    )

    max_latest_minus_decision = float(
        good[
            "latest_minus_decision_s"
        ].max()
    )
else:
    max_latest_minus_decision = np.nan

causal_latest_pass = (
    len(good) == 2521
    and max_latest_minus_decision <= 1e-9
)

print(
    "Max latest_used - decision s:",
    max_latest_minus_decision,
)

print(
    "Latest-used causal gate:",
    causal_latest_pass,
)

# Negative test:
# For a deterministic sample of cycles, arbitrarily overwrite all samples AFTER decision.
# Because the reconstruction caps available samples at decision time, features must remain identical.
sample_idx = np.linspace(
    0,
    len(work) - 1,
    min(50, len(work)),
    dtype=int,
)

negative_rows = []

for idx in sample_idx:

    row = work.iloc[idx]

    if pd.isna(
        row["discharge_sequence_index"]
    ):
        continue

    battery = row["battery_id"]
    cycles = get_battery_cycles(battery)

    cyc = cycles[
        int(
            row["discharge_sequence_index"]
        )
    ]

    (
        grid,
        v0,
        i0,
        t0,
        support0,
    ) = reconstruct_window(
        cyc,
        float(row["window_start_s"]),
        float(row["window_end_s"]),
        float(row["decision_time_s"]),
    )

    f0 = extract_44(
        v0,
        i0,
        t0,
        DURATION_S,
    )

    # Copy and perturb future.
    altered = {
        k: (
            np.array(v, copy=True)
            if isinstance(v, np.ndarray)
            else v
        )
        for k, v in cyc.items()
    }

    future_mask = (
        altered["time"]
        > float(
            row["decision_time_s"]
        )
        + EPS
    )

    altered["voltage"][future_mask] += 123.456
    altered["current"][future_mask] -= 77.0
    altered["temperature"][future_mask] += 55.0

    (
        grid,
        v1,
        i1,
        t1,
        support1,
    ) = reconstruct_window(
        altered,
        float(row["window_start_s"]),
        float(row["window_end_s"]),
        float(row["decision_time_s"]),
    )

    f1 = extract_44(
        v1,
        i1,
        t1,
        DURATION_S,
    )

    a = np.asarray(
        [f0[n] for n in feature_order],
        dtype=float,
    )

    b = np.asarray(
        [f1[n] for n in feature_order],
        dtype=float,
    )

    d = float(
        np.max(
            np.abs(a - b)
        )
    )

    negative_rows.append({
        "cycle_uid": row["cycle_uid"],
        "battery_id": battery,
        "max_feature_diff_after_future_perturbation": d,
        "pass": d == 0.0,
    })


negative = pd.DataFrame(
    negative_rows
)

display(negative)

negative_pass = (
    len(negative) > 0
    and negative["pass"].all()
)

print(
    "Future perturbation negative test:",
    negative_pass,
)


Max latest_used - decision s: 7.105427357601002e-15
Latest-used causal gate: True


,cycle_uid,battery_id,max_feature_diff_after_future_perturbation,pass
0,B0005_D0001,B0005,0.0,True
1,B0005_D0052,B0005,0.0,True
2,B0005_D0103,B0005,0.0,True
3,B0005_D0155,B0005,0.0,True
4,B0006_D0038,B0006,0.0,True
5,B0006_D0090,B0006,0.0,True
6,B0006_D0141,B0006,0.0,True
7,B0007_D0025,B0007,0.0,True
8,B0007_D0076,B0007,0.0,True
9,B0007_D0127,B0007,0.0,True


Future perturbation negative test: True


In [12]:

PROV_PATH = OUT / "online_cycle_raw_provenance_v2.csv"

prov.to_csv(
    PROV_PATH,
    index=False,
    encoding="utf-8-sig",
)

cycle_map.to_csv(
    OUT / "raw_discharge_cycle_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

raw_map.to_csv(
    OUT / "raw_battery_file_map.csv",
    index=False,
    encoding="utf-8-sig",
)

feature_diffs.to_csv(
    OUT / "raw_to_44_feature_parity_by_cycle.csv",
    index=False,
    encoding="utf-8-sig",
)

negative.to_csv(
    OUT / "future_perturbation_negative_test.csv",
    index=False,
    encoding="utf-8-sig",
)

timing_sources.to_csv(
    OUT / "timing_metadata_sources.csv",
    index=False,
    encoding="utf-8-sig",
)

# Copy into GitHub repo if it exists.
if REPO.exists():
    metadata_dir = REPO / "data/metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(
        PROV_PATH,
        metadata_dir / "online_cycle_raw_provenance_v2.csv",
    )

    shutil.copy2(
        OUT / "future_perturbation_negative_test.csv",
        metadata_dir / "future_perturbation_negative_test.csv",
    )

    shutil.copy2(
        OUT / "raw_to_44_feature_parity_by_cycle.csv",
        metadata_dir / "raw_to_44_feature_parity_by_cycle.csv",
    )

print("Saved:", PROV_PATH)


Saved: ${PROJECT_ROOT}\results\29_F5_online_raw_provenance_reconstruction\online_cycle_raw_provenance_v2.csv


In [13]:

status_counts = (
    prov.groupby(
        "provenance_status"
    ).size()
)

pass_rows = int(
    (
        prov[
            "provenance_status"
        ]
        == "PASS"
    ).sum()
)

full_rows_pass = pass_rows == 2521

raw_hash_pass = (
    prov["raw_source_sha256"]
    .notna()
    .sum()
    == 2521
)

indices_pass = (
    prov["raw_sample_indices"]
    .notna()
    .sum()
    == 2521
)

latest_pass = (
    prov["latest_used_timestamp_s"]
    .notna()
    .sum()
    == 2521
)

feature_parity_pass = (
    len(feature_diffs) == 2521
    and float(
        feature_diffs[
            "max_abs_diff"
        ].max()
    ) <= 1e-5
)

future_support_stored_pass = False

if (
    "future_support_s"
    in prov.columns
    and prov["future_support_s"].notna().all()
):
    max_future_support = float(
        pd.to_numeric(
            prov["future_support_s"]
        ).max()
    )
    future_support_stored_pass = (
        max_future_support <= 1e-9
    )
else:
    max_future_support = np.nan

checks = pd.DataFrame([
    {
        "check":
            "2521 cycles have PASS provenance",
        "pass":
            full_rows_pass,
    },
    {
        "check":
            "raw_source_sha256 complete",
        "pass":
            raw_hash_pass,
    },
    {
        "check":
            "raw_sample_indices complete",
        "pass":
            indices_pass,
    },
    {
        "check":
            "latest_used_timestamp_s complete",
        "pass":
            latest_pass,
    },
    {
        "check":
            "raw→44 feature parity <=1e-5 for all cycles",
        "pass":
            feature_parity_pass,
    },
    {
        "check":
            "latest-used timestamp never exceeds decision",
        "pass":
            causal_latest_pass,
    },
    {
        "check":
            "stored future_support is zero",
        "pass":
            future_support_stored_pass,
    },
    {
        "check":
            "future perturbation negative test passes",
        "pass":
            negative_pass,
    },
])

display(checks)

f5_pass = bool(
    checks["pass"].all()
)

print("=" * 100)
print("F5 ONLINE RAW PROVENANCE RECONSTRUCTION — FINAL SUMMARY")
print("=" * 100)

print("Protocol:", PROTOCOL_ID)
print("Rows:", len(prov))
print("PASS provenance rows:", pass_rows)
print("Raw files hashed:", prov["raw_source_file"].nunique())

print()
print("MISSING FIELDS")
print(
    "raw_source_sha256 missing:",
    int(
        prov["raw_source_sha256"]
        .isna()
        .sum()
    ),
)
print(
    "raw_sample_indices missing:",
    int(
        prov["raw_sample_indices"]
        .isna()
        .sum()
    ),
)
print(
    "latest_used_timestamp_s missing:",
    int(
        prov["latest_used_timestamp_s"]
        .isna()
        .sum()
    ),
)

print()
print("PARITY")
print(
    "Global raw→44 max abs feature diff:",
    (
        feature_diffs[
            "max_abs_diff"
        ].max()
        if len(feature_diffs)
        else np.nan
    ),
)

print()
print("CAUSALITY")
print(
    "Max latest_used - decision s:",
    max_latest_minus_decision,
)
print(
    "Max stored future_support s:",
    max_future_support,
)
print(
    "Future perturbation negative test:",
    negative_pass,
)

print()
print("STATUS COUNTS")
print(status_counts.to_string())

print()
print(
    "F5 PROVENANCE STATUS:",
    "PASS"
    if f5_pass
    else "PARTIAL/FAIL",
)

if not f5_pass:
    print()
    print("Review non-PASS rows:")
    display(
        prov[
            prov["provenance_status"]
            != "PASS"
        ].head(50)
    )


,check,pass
0,2521 cycles have PASS provenance,True
1,raw_source_sha256 complete,True
2,raw_sample_indices complete,True
3,latest_used_timestamp_s complete,True
4,raw→44 feature parity <=1e-5 for all cycles,True
5,latest-used timestamp never exceeds decision,True
6,stored future_support is zero,True
7,future perturbation negative test passes,True


F5 ONLINE RAW PROVENANCE RECONSTRUCTION — FINAL SUMMARY
Protocol: ONLINE_FIXED_START_CAUSAL_10S_V1
Rows: 2521
PASS provenance rows: 2521
Raw files hashed: 34

MISSING FIELDS
raw_source_sha256 missing: 0
raw_sample_indices missing: 0
latest_used_timestamp_s missing: 0

PARITY
Global raw→44 max abs feature diff: 0.0

CAUSALITY
Max latest_used - decision s: 7.105427357601002e-15
Max stored future_support s: 0.0
Future perturbation negative test: True

STATUS COUNTS
provenance_status
PASS    2521

F5 PROVENANCE STATUS: PASS
